# Exercise: Generating Images with Reverse Diffusion

In this exercise, you'll implement the reverse diffusion process to generate MNIST digits from pure noise. You've already trained a diffusion model - now it's time to use it!

## Learning Objectives
- Reconstruct the noise schedule used during training
- Implement the DDPM reverse sampling algorithm
- Generate new images by denoising from random Gaussian noise

## What You'll Complete
1. **TODO 1**: Rebuild the noise schedule (betas, alphas, and cumulative products)
2. **TODO 2**: Implement the `sample_ddpm()` function for reverse diffusion

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import math

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(" Imports complete")

## Model Architecture Definitions

These are the same components from your training notebook. No changes needed here.

In [ ]:
class TimeEmbedding(nn.Module):
    """Sinusoidal time embedding for timestep conditioning."""

    def __init__(self, embedding_dim=128):
        super().__init__()
        self.embedding_dim = embedding_dim

    def forward(self, timestep):
        device = timestep.device
        half_dim = self.embedding_dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half_dim, device=device) / half_dim
        )
        args = timestep[:, None].float() * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResidualBlock(nn.Module):
    """Residual block with timestep conditioning."""

    def __init__(self, in_channels, out_channels, time_embedding_dim=128):
        super().__init__()
        self.norm1 = nn.GroupNorm(num_groups=32, num_channels=in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(num_groups=32, num_channels=out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_embedding_dim, out_channels),
            nn.SiLU(),
            nn.Linear(out_channels, out_channels),
        )
        if in_channels != out_channels:
            self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.skip = nn.Identity()

    def forward(self, x, time_embedding):
        h = self.norm1(x)
        h = F.silu(h)
        h = self.conv1(h)
        time_scale_shift = self.time_mlp(time_embedding)[:, :, None, None]
        h = h * time_scale_shift
        h = self.norm2(h)
        h = F.silu(h)
        h = self.conv2(h)
        return h + self.skip(x)


class SimpleUNet(nn.Module):
    """Simple U-Net for MNIST noise prediction."""

    def __init__(self, image_channels=1, base_channels=64, time_embedding_dim=128):
        super().__init__()
        self.time_embedding = TimeEmbedding(time_embedding_dim)
        self.init_conv = nn.Conv2d(
            image_channels, base_channels, kernel_size=3, padding=1
        )

        # Encoder
        self.down_conv1 = nn.Conv2d(
            base_channels, base_channels, kernel_size=4, stride=2, padding=1
        )
        self.down_res1 = ResidualBlock(
            base_channels, base_channels * 2, time_embedding_dim
        )

        self.down_conv2 = nn.Conv2d(
            base_channels * 2, base_channels * 2, kernel_size=4, stride=2, padding=1
        )
        self.down_res2 = ResidualBlock(
            base_channels * 2, base_channels * 2, time_embedding_dim
        )

        # Middle
        self.middle_res = ResidualBlock(
            base_channels * 2, base_channels * 2, time_embedding_dim
        )

        # Decoder
        self.up_conv2 = nn.ConvTranspose2d(
            base_channels * 2, base_channels * 2, kernel_size=4, stride=2, padding=1
        )
        self.up_res2 = ResidualBlock(
            base_channels * 2, base_channels, time_embedding_dim
        )

        self.up_conv1 = nn.ConvTranspose2d(
            base_channels, base_channels, kernel_size=4, stride=2, padding=1
        )
        self.up_res1 = ResidualBlock(base_channels, base_channels, time_embedding_dim)

        # Final output
        self.final_norm = nn.GroupNorm(num_groups=32, num_channels=base_channels)
        self.final_conv = nn.Conv2d(
            base_channels, image_channels, kernel_size=3, padding=1
        )

    def forward(self, x, timestep):
        time_emb = self.time_embedding(timestep)
        h = self.init_conv(x)

        h = self.down_conv1(h)
        h = self.down_res1(h, time_emb)

        h = self.down_conv2(h)
        h = self.down_res2(h, time_emb)

        h = self.middle_res(h, time_emb)

        h = self.up_conv2(h)
        h = self.up_res2(h, time_emb)

        h = self.up_conv1(h)
        h = self.up_res1(h, time_emb)

        h = self.final_norm(h)
        h = F.silu(h)
        h = self.final_conv(h)
        return h


print(" Model architecture defined")

## TODO 1: Rebuild the Noise Schedule

The noise schedule **must exactly match** what was used during training. Otherwise, the model won't be able to denoise properly!

You need to compute:
1. **betas**: Linear schedule from 0.0001 to 0.02 over 1000 timesteps
2. **alphas**: Signal retention factors (1 - betas)
3. **alphas_cumprod**: Cumulative product of alphas (ᾱ_t)
4. **alphas_cumprod_prev**: Shifted version with a leading 1.0

### Hints:
- Use `torch.linspace()` for the linear beta schedule
- Use `torch.cumprod()` for cumulative products
- Use `torch.cat()` to prepend a value to a tensor

### Why This Matters:
These pre-computed values allow efficient reverse diffusion. The cumulative products tell us exactly how much noise exists at each timestep, which is critical for the denoising formula.

In [ ]:
# TODO 1: Rebuild the noise schedule
num_timesteps = 1000

# Step 1: Create linear beta schedule from 0.0001 to 0.02

betas = None 

# Step 2: Compute alphas (signal retention factors)

alphas = None  

# Step 3: Compute cumulative product of alphas

alphas_cumprod = None  

# Step 4: Create shifted version with leading 1.0

alphas_cumprod_prev = None  

# Verify your implementation
if betas is not None and alphas is not None and alphas_cumprod is not None and alphas_cumprod_prev is not None:
    print(" Noise schedule variables defined")
    print(f"  betas shape: {betas.shape}, range: [{betas.min():.4f}, {betas.max():.4f}]")
    print(f"  alphas_cumprod shape: {alphas_cumprod.shape}")
    print(f"  alphas_cumprod[0] = {alphas_cumprod[0]:.4f} (should be ~0.9999)")
    print(f"  alphas_cumprod[-1] = {alphas_cumprod[-1]:.6f} (should be very small)")
    print(f"  alphas_cumprod_prev[0] = {alphas_cumprod_prev[0]:.1f} (should be 1.0)")
else:
    print(" Complete TODO 1 to define the noise schedule")

In [ ]:
# Load trained model
model = SimpleUNet(image_channels=1, base_channels=64, time_embedding_dim=128).to(device)

checkpoint_path = Path("../../../model_checkpoints/diffusion_model_lesson16.pt")
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

epochs_trained = checkpoint.get("epoch", "?")
print(f"✓ Loaded model trained for {epochs_trained} epochs")
print(f"  From: {checkpoint_path.resolve()}")

## TODO 2: Implement DDPM Sampling

Now for the exciting part - generating images from pure noise!

### The Reverse Diffusion Process:
Starting from random Gaussian noise x_T, we iteratively denoise:
```
x_T → x_{T-1} → x_{T-2} → ... → x_1 → x_0
```

At each timestep t, the denoising formula is:
```
x_{t-1} = (x_t - (1-α_t)/√(1-ᾱ_t) * ε_pred) / √α_t + √β̃_t * z
```

Where:
- **ε_pred**: Noise predicted by the U-Net
- **α_t**: Current alpha value
- **ᾱ_t**: Cumulative alpha product
- **β̃_t**: Posterior variance = (1 - ᾱ_{t-1}) / (1 - ᾱ_t) * β_t
- **z**: Random noise (only added when t > 0)

### Your Task:
Complete the `sample_ddpm()` function by:
1. Starting with pure noise
2. Looping backward through timesteps (T-1 down to 0)
3. Using the model to predict noise
4. Computing the posterior variance
5. Applying the denoising formula
6. Adding noise (except at t=0)

In [ ]:
@torch.no_grad()
def sample_ddpm(model, num_samples=16):
    """Generate images via reverse diffusion: x_T → x_0."""
    model.eval()
    
    # TODO 2.1: Initialize x with pure Gaussian noise
    x = None  
    
    # TODO 2.2: Loop backward through timesteps
   
    for t_idx in []:  
        
        # Create timestep tensor for the batch
        t = torch.full((num_samples,), t_idx, dtype=torch.long, device=device)
        
        # TODO 2.3: Get noise prediction from the model
       
        pred_noise = None  
        
        # TODO 2.4: Get coefficients for this timestep
   
        alpha = None  
        alpha_cum = None  
        alpha_cum_prev = None  
        
        # TODO 2.5: Compute posterior variance
     
        posterior_var = None  
        
        # TODO 2.6: Apply denoising formula
    
        x = None 
        
        # TODO 2.7: Add noise for all timesteps except the last (t=0)

        pass  
    
    # Final output to valid image range
    return torch.clamp(x, -1.0, 1.0)


print(" Sampling function defined")